# Synthetic Chinchilla-E validation (A100)

Known Zipf corpus → train three tiny models (~0.8–2.8M params) → triangulate → compare to analytic **H_true**.

**Before you run:** Runtime → **GPU** (A100 recommended). Upload `small-lm-lab/scripts/` to Colab (see next cell) or set `REPO` to your Drive path.

Pre-registration: `docs/PREREGISTER_synthetic_chinchilla_e.md`

In [ ]:
# @title 1) GPU check + minimal deps
!pip install -q scipy

import os, subprocess, textwrap
import torch

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Device: {gpu}")
if not torch.cuda.is_available():
    print("WARNING: no GPU — this will be slow. Use Runtime → Change runtime type → GPU.")

In [ ]:
# @title 2) Point at your code (edit REPO if needed)
# Upload the folder small-lm-lab/scripts/ via Colab file browser → /content/small-lm-lab/scripts/
# Or mount Drive and set REPO to your copy.

from google.colab import drive

USE_DRIVE = False  # @param {type:"boolean"}
REPO = "/content/small-lm-lab"  # @param {type:"string"}
DRIVE_REPO = "/content/drive/MyDrive/small-lm-lab"  # @param {type:"string"}

if USE_DRIVE:
    drive.mount("/content/drive")
    REPO = DRIVE_REPO

SCRIPTS = os.path.join(REPO, "scripts")
for name in ("synthetic_chinchilla_e_validation.py", "owt_chinchilla_e.py"):
    path = os.path.join(SCRIPTS, name)
    assert os.path.isfile(path), f"Missing {path} — upload scripts/ or fix REPO"

os.chdir(SCRIPTS)
print(f"OK: {SCRIPTS}")

In [ ]:
# @title 3) Run validation (~15–25 min on A100)
import os

PRESET = "a100_fast"  # @param ["a100_fast", "smoke", "full"]
OUT_DIR = "/content/synthetic_chinchilla"  # fast local disk

os.environ["SYNTH_PRESET"] = PRESET
os.environ["SYNTH_CHINCHILLA_DIR"] = OUT_DIR

!python synthetic_chinchilla_e_validation.py

In [ ]:
# @title 4) Results
from IPython.display import Markdown, display
import json

OUT_DIR = "/content/synthetic_chinchilla"
report_path = f"{OUT_DIR}/validation_report.txt"
tri_path = f"{OUT_DIR}/triangulation.json"

with open(report_path, encoding="utf-8") as f:
    report = f.read()
print(report)

with open(tri_path, encoding="utf-8") as f:
    tri = json.load(f)
v = tri["validation"]
passed = v["primary_pass"]
display(Markdown(
    f"**Primary pass (P1 & P2):** {'PASS' if passed else 'FAIL'}  \n"
    f"H_true = {v['H_true_nats']:.4f} nats | E_true (free) = {v['E_true_free_alpha']:.4f} | "
    f"delta = {v['delta_free']:.4f} nats"
))

In [ ]:
# @title 5) Optional — copy results to Drive
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
DRIVE_OUT = "/content/drive/MyDrive/synthetic_chinchilla_results"  # @param {type:"string"}

import os
import shutil

if SAVE_TO_DRIVE:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    shutil.copytree("/content/synthetic_chinchilla", DRIVE_OUT, dirs_exist_ok=True)
    print(f"Saved to {DRIVE_OUT}")
else:
    print("Skipped Drive copy.")